# Capítulo 3 · Salidas estructuradas

Extraer de cada correo el nombre del cliente, el número de pedido, el producto y la categoría en un JSON que un programa pueda leer: cinco formas de pedirlo, validación con Pydantic, reintentos y una plantilla con generación restringida.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*. Con GPU, el cuaderno usa Qwen2.5-1.5B-Instruct; sin ella, Qwen3-0.6B. El cuaderno completo tarda unos minutos.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

📘 El libro, en Amazon: [tapa blanda](https://www.amazon.es/dp/B0HMXCBCTN) · [Kindle](https://www.amazon.es/dp/B0HMXG3NXM)

## 1. El modelo y los datos

La primera vez se descarga el modelo y el archivo con los 24 correos y sus datos correctos.

In [ ]:
"""Capítulo 3: salidas estructuradas. Modelo abierto, en GPU o CPU."""
import json
import re
import unicodedata
import urllib.request
from pathlib import Path
from typing import Literal, Optional

import torch
from pydantic import BaseModel, ConfigDict, ValidationError, constr
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen2.5-1.5B-Instruct"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-0.6B"             # sin GPU: más pequeño y más rápido
MODELO = None                              # None: elegir según el entorno
DATOS = Path("datos") / "correos_extraccion.json"
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/"



def cargar(nombre=None):
    """Carga el modelo y su tokenizador en la GPU si la hay; si no, en la CPU."""
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    if nombre is None:
        nombre = MODELO_GPU if dispositivo == "cuda" else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(nombre, dtype=torch.float32)
    modelo.to(dispositivo).eval()
    print(f"Modelo {nombre} en {dispositivo}")
    return tok, modelo


def cargar_correos():
    """Los correos con sus datos correctos; en Colab, se descargan."""
    if not DATOS.exists():
        DATOS.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + DATOS.as_posix(), DATOS)
    return json.loads(DATOS.read_text(encoding="utf-8"))

In [ ]:
tok, modelo = cargar(MODELO)
correos = cargar_correos()
print(len(correos), "correos. El primero:")
print(json.dumps(correos[0], ensure_ascii=False, indent=1))

## 2. Conversar

`responder` recibe la conversación completa, porque el reintento necesita añadirle turnos.

In [ ]:
def mensajes(sistema, usuario, ejemplos=()):
    """Sistema, ejemplos resueltos (pares pregunta-respuesta) y la pregunta."""
    lista = [{"role": "system", "content": sistema}] if sistema else []
    for pregunta, respuesta in ejemplos:
        lista.append({"role": "user", "content": pregunta})
        lista.append({"role": "assistant", "content": respuesta})
    return lista + [{"role": "user", "content": usuario}]


def texto_conversacion(conversacion):
    """La conversación con la plantilla de chat, lista para que responda el modelo."""
    return tok.apply_chat_template(conversacion, tokenize=False,
                                   add_generation_prompt=True,
                                   enable_thinking=False)


def responder(conversacion, max_tokens=200):
    """Respuesta del modelo a una conversación, con temperatura 0."""
    entrada = tok(texto_conversacion(conversacion),
                  return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    respuesta = tok.decode(nuevos, skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", respuesta, flags=re.S).strip()

## 3. El esquema y la validación

In [ ]:
CATEGORIAS = ("pedido", "devolucion", "averia", "consulta", "queja")


class Extraccion(BaseModel):
    """Los datos que queremos de cada correo."""
    model_config = ConfigDict(extra="forbid")      # ninguna clave de más
    nombre: Optional[str]
    pedido: Optional[constr(pattern=r"^CN-\d{5}$")]
    producto: Optional[str]
    categoria: Literal[CATEGORIAS]


def leer_json(texto):
    """El objeto JSON de la respuesta, aunque venga con texto alrededor."""
    inicio, fin = texto.find("{"), texto.rfind("}")
    if inicio == -1 or fin < inicio:
        raise ValueError("la respuesta no contiene ningún objeto JSON")
    return json.loads(texto[inicio:fin + 1])


MOTIVOS = {"missing": "falta este campo",
           "extra_forbidden": "este campo no está permitido",
           "string_pattern_mismatch": "debe tener el formato CN-12345",
           "literal_error": "debe ser una de: " + ", ".join(CATEGORIAS),
           "string_type": "debe ser un texto o null"}


def errores(excepcion):
    """Los errores de validación, explicados en español."""
    if isinstance(excepcion, ValidationError):
        return "; ".join(f"«{e['loc'][0]}»: {MOTIVOS.get(e['type'], e['msg'])}"
                         for e in excepcion.errors())
    return f"no es un JSON válido ({excepcion})"

Prueba la validación con datos escritos a mano: uno correcto y otro con tres errores.

In [ ]:
print(Extraccion(nombre="Laura", pedido="CN-10482", producto="bicicleta", categoria="devolucion"))
try:
    Extraccion(nombre="Laura", pedido="10482", producto="bicicleta", categoria="cambio", urgente=True)
except ValidationError as e:
    print(errores(e))

## 4. Pedir, validar y reintentar

In [ ]:
PEDIR_JSON = ("Extrae de este correo de un cliente de Ciclos Norte, una tienda "
              "de bicicletas, el nombre del cliente, el número de pedido, el "
              "producto y la categoría del correo. Responde en JSON.")


ESQUEMA = (
    "Extrae los datos de un correo de un cliente de Ciclos Norte, una tienda "
    "de bicicletas. Responde solo con un objeto JSON, sin ningún texto "
    "alrededor, con exactamente estas cuatro claves:\n"
    '- "nombre": el nombre del cliente tal como firma, o null si no lo da.\n'
    '- "pedido": el número de pedido con el formato CN-12345, o null.\n'
    '- "producto": el producto del que habla, en pocas palabras, o null.\n'
    '- "categoria": una de estas cinco:\n'
    "  pedido (seguimiento o cambios de un pedido ya hecho),\n"
    "  devolucion (devolver o cambiar un producto, o su reembolso),\n"
    "  averia (un producto no funciona o está roto),\n"
    "  consulta (preguntas antes de comprar),\n"
    "  queja (protestas por el trato o el servicio).\n"
    "Si un dato no aparece en el correo, escribe null: no lo inventes.")


EJEMPLOS = [
    ("Buenas, el sillín del pedido 10230 se ha rajado al mes. ¿Me lo "
     "cambiáis? Un saludo, Rosa Díaz",
     '{"nombre": "Rosa Díaz", "pedido": "CN-10230", "producto": "sillín", '
     '"categoria": "averia"}'),
    ("Llevo dos días esperando una respuesta vuestra. Así no se puede.",
     '{"nombre": null, "pedido": null, "producto": null, '
     '"categoria": "queja"}')]

In [ ]:
def extraer(correo, sistema, ejemplos=(), reintentos=0):
    """Pide los datos y, si no son válidos, vuelve a pedirlos con los errores."""
    conversacion = mensajes(sistema, correo, ejemplos)
    for intento in range(reintentos + 1):
        respuesta = responder(conversacion)
        try:
            return respuesta, Extraccion(**leer_json(respuesta)), intento
        except (ValueError, TypeError, ValidationError) as e:
            conversacion += [
                {"role": "assistant", "content": respuesta},
                {"role": "user", "content": f"Tu respuesta no es válida: "
                 f"{errores(e)}. Corrígela y responde solo con el JSON."}]
    return respuesta, None, intento

In [ ]:
correo = correos[3]["correo"]
print(correo, "\n")
print("Pedir JSON:\n", extraer(correo, PEDIR_JSON)[0], "\n")
print("Con esquema y ejemplos:\n", extraer(correo, ESQUEMA, EJEMPLOS)[0])

## 5. Rellenar una plantilla (generación restringida)

El programa escribe la estructura y el modelo solo los valores, con las opciones limitadas.

In [ ]:
def ids(texto):
    """Los números de los tokens de un texto, sin tokens especiales."""
    return tok(texto, add_special_tokens=False)["input_ids"]


def elegir(prefijo, opciones):
    """La opción más probable como continuación del prefijo."""
    base = ids(prefijo)
    puntos = []
    for opcion in opciones:
        entrada = torch.tensor([base + ids(opcion)], device=modelo.device)
        with torch.no_grad():
            logp = torch.log_softmax(modelo(entrada).logits[0, :-1], dim=-1)
        posiciones = range(len(base) - 1, entrada.shape[1] - 1)
        puntos.append(sum(logp[i, entrada[0, i + 1]].item() for i in posiciones))
    return opciones[puntos.index(max(puntos))]


def escribir(prefijo, permitidos=None, max_tokens=15):
    """Genera texto libre hasta la comilla de cierre, o solo con ciertos tokens."""
    entrada = torch.tensor([ids(prefijo)], device=modelo.device)
    limite = (lambda lote, generados: permitidos) if permitidos else None
    with torch.no_grad():
        salida = modelo.generate(entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, prefix_allowed_tokens_fn=limite,
                                 pad_token_id=tok.eos_token_id)
    return tok.decode(salida[0, entrada.shape[1]:]).split('"')[0]


def rellenar(correo):
    """El programa escribe la estructura; el modelo, solo los valores."""
    texto = texto_conversacion(mensajes(ESQUEMA, correo, EJEMPLOS))
    datos = {}
    for campo in ("nombre", "pedido", "producto", "categoria"):
        texto += ("{" if campo == "nombre" else ", ") + f'"{campo}":'
        if campo == "categoria":
            valor = elegir(texto, [f' "{c}"' for c in CATEGORIAS])[2:-1]
        elif elegir(texto, [" null", ' "']) == " null":
            valor = None
        elif campo == "pedido":
            digitos = [ids(str(d))[0] for d in range(10)]
            valor = "CN-" + escribir(texto + ' "CN-', digitos, max_tokens=5)
        else:
            valor = escribir(texto + ' "').strip()
        datos[campo] = valor
        texto += " null" if valor is None else f' "{valor}"'
    return json.dumps(datos, ensure_ascii=False), Extraccion(**datos), 0

In [ ]:
print(rellenar(correos[3]["correo"])[0])

## 6. Comparar con los datos correctos

In [ ]:
def sin_tildes(texto):
    """Minúsculas y sin tildes, para comparar sin preocuparse de ellas."""
    return "".join(c for c in unicodedata.normalize("NFD", texto.lower().strip())
                   if unicodedata.category(c) != "Mn")


def acierta(campo, obtenido, correcto):
    """¿Coincide un campo? En el producto basta con que contenga la palabra clave."""
    if correcto is None or obtenido in (None, "", "null"):
        return correcto is None and obtenido in (None, "", "null")
    obtenido, correcto = sin_tildes(str(obtenido)), sin_tildes(correcto)
    return correcto in obtenido if campo == "producto" else obtenido == correcto


CAMPOS = ("nombre", "pedido", "producto", "categoria")


def evaluar(correos, forma):
    """Extrae los datos de todos los correos y cuenta lo que sale bien."""
    filas = []
    for c in correos:
        respuesta, datos, intentos = forma(c["correo"])
        try:
            leido = json.loads(respuesta)
            se_lee = True
        except ValueError:
            se_lee = False
            try:
                leido = leer_json(respuesta)
            except ValueError:
                leido = {}
        if not isinstance(leido, dict):
            leido = {}
        fila = {"respuesta": respuesta, "se_lee": se_lee, "valido": datos is not None,
                "reintentos": intentos}
        fila.update({k: k in leido and acierta(k, leido[k], c[k]) for k in CAMPOS})
        filas.append(fila)
    return filas

In [ ]:
FORMAS = {
    "1. Pedir JSON": lambda correo: extraer(correo, PEDIR_JSON),
    "2. Con esquema": lambda correo: extraer(correo, ESQUEMA),
    "3. Con ejemplos": lambda correo: extraer(correo, ESQUEMA, EJEMPLOS),
    "4. Validar y reintentar": lambda correo: extraer(correo, ESQUEMA, EJEMPLOS,
                                                      reintentos=2),
    "5. Rellenar plantilla": rellenar,
}


def tabla(resultados):
    """Una fila por forma: JSON que se lee tal cual, válidos y campos correctos."""
    columnas = ("se_lee", "valido") + CAMPOS
    print(f"{'Forma':25s}" + "".join(f"{c[:9]:>10s}" for c in columnas))
    for forma, filas in resultados.items():
        print(f"{forma:25s}" + "".join(f"{sum(f[c] for f in filas):>7d}/{len(filas)}"
                                       for c in columnas))

In [ ]:
resultados = {}
for forma, funcion in FORMAS.items():
    resultados[forma] = evaluar(correos, funcion)
    print(forma, "terminada")
tabla(resultados)

Lee las respuestas en las que falla algún campo de la mejor forma:

In [ ]:
for fila, correo in zip(resultados["5. Rellenar plantilla"], correos):
    fallos = [campo for campo in CAMPOS if not fila[campo]]
    if fallos:
        print(fallos, fila["respuesta"])
        print("   correcto:", {campo: correo[campo] for campo in CAMPOS})

## Ejercicios

1. Explica con tus palabras la diferencia entre leer una respuesta, validarla y comprobar que es correcta. ¿Qué herramienta del capítulo se ocupa de cada cosa?

2. Escribe a mano tres respuestas que `json.loads` no pueda leer y tres que se lean pero no cumplan el esquema. Comprueba con `Extraccion` qué errores da cada una.

3. Añade al esquema un campo `urgente`, de tipo `bool`, que indique si el cliente pide una solución rápida o está enfadado. ¿Cómo lo describirías en la instrucción? ¿Y en la plantilla?

4. En la tercera forma, quita el segundo ejemplo, el del correo sin datos. ¿Vuelven los nombres inventados?

5. Cambia `extraer` para que el mensaje del reintento incluya también los campos que estaban bien y pida que no los cambie. ¿Se conservan ahora el nombre y el pedido de Ana?

6. El programa normaliza los números de pedido pidiendo al modelo el formato `CN-12345`. Escribe una función de Python que lo haga a partir de lo que escribe el cliente (`10377`, `cn-11020`, `CN 10655`…) y compárala con el modelo. ¿Qué es más fiable? ¿Cuándo usarías cada uno?

7. En `rellenar`, el nombre se escribe libremente. Restringe sus tokens a los que aparecen en el correo, con `prefix_allowed_tokens_fn`. ¿Mejora? ¿Qué problemas aparecen?

8. Mide cuánto tarda cada forma por correo. ¿Compensa el tiempo de la plantilla frente a validar y reintentar?

9. Añade cinco correos de prueba más difíciles: dos productos en un mismo correo, una firma con apodo o un número de pedido mal escrito por el cliente (`CN-1037`). ¿Qué debería responder el modelo en cada caso? ¿Lo hace?

10. **Ficha del cliente.** Usa la mejor forma para extraer los datos de los 40 correos de prueba del capítulo 2 y compara la categoría extraída con la de ese capítulo. ¿Es mejor clasificar por separado o junto con los demás datos?

## Referencias

Modelos [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) y [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), de Qwen (Alibaba), con licencia Apache 2.0. [Pydantic](https://docs.pydantic.dev). Generación restringida: [Willard y Louf (2023)](https://arxiv.org/abs/2307.09702). Lecciones 11.03 y 05.20 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La tienda Ciclos Norte y sus correos son ficticios. Código y experimentos propios.